# 09 — Project1 Submission Dynamics: Animated Plotly Chart

## What this notebook does

Recreates the same visualization as notebook 08 — submission dynamics
for `project1` — but as an interactive animated chart using Plotly,
where the timeline unfolds frame by frame.

## Data

- **Source:** `checking-logs.sqlite` → table `checker`
- **Filtered:** only real students (`uid LIKE 'user_%'`), only `project1`,
  only successful submissions (`status = 'ready'`)
- **Aggregation:** maximum `numTrials` per user per day, forward-filled
  to create a continuous timeline

## Key steps

1. Query `uid`, `timestamp` and `numTrials` for project1 submissions
2. Group by user and date, take max `numTrials` per day
3. Pivot into wide format, forward-fill missing dates, melt back to long format
4. Build a Plotly figure with one line per user
5. Generate animation frames — each frame adds one more date to the chart
6. Add a Play button to control the animation

## Result

An interactive animated line chart where each frame represents one day,
showing how submission attempts accumulated over the course of project1.
Provides a more engaging and dynamic view of the same data compared to
the static seaborn version in notebook 08.

## Tools

`pandas` · `sqlite3` · `plotly` · `numpy`

In [1]:
import pandas as pd
import sqlite3
import plotly.graph_objects as go
import numpy as np

conn = sqlite3.connect('../data/checking-logs.sqlite')

In [2]:
sea_data = pd.read_sql("""
                       SELECT c.uid, c.timestamp, c.numTrials
                       FROM checker AS c
                       WHERE c.labname = 'project1'
                       AND c.uid LIKE 'user_%'
                       AND status = 'ready'
                       """,
                       conn)

sea_data['timestamp'] = pd.to_datetime(sea_data['timestamp'])
sea_data['date'] = sea_data['timestamp'].dt.date
sea_data_grouped = sea_data.groupby(['uid', 'date'])['numTrials'].max().reset_index()
sea_data_grouped = sea_data_grouped.pivot(index='date', columns='uid', values='numTrials').ffill().fillna(0).reset_index().melt(id_vars='date', value_name='numTrials')
sea_data_grouped.head(1)

,date,uid,numTrials
0,2020-04-17,user_1,0.0


In [3]:
dates = sorted(sea_data_grouped['date'].unique()) #numpy
users = sea_data_grouped['uid'].unique()

In [4]:
fig = go.Figure()

# Начальные линии для каждого пользователя
for user in users:
    user_data = sea_data_grouped[sea_data_grouped['uid'] == user]
    fig.add_trace(go.Scatter(
        x=user_data['date'],
        y=user_data['numTrials'],
        mode='lines+markers',
        name=user,
        line=dict(width=3)
    ))

# Кадры анимации
frames = []
for date in dates:
    frame_data = []
    for user in users:
        user_data = sea_data_grouped[
            (sea_data_grouped['uid'] == user) & 
            (sea_data_grouped['date'] <= date)
        ]
        frame_data.append(go.Scatter(
            x=user_data['date'],
            y=user_data['numTrials'],
            mode='lines+markers',
            line=dict(width=3)
        ))
    frames.append(go.Frame(data=frame_data, name=str(date)))

fig.frames = frames

fig.update_layout(
    title=dict(text='Dynamic of commits per user in project1', font=dict(size=30)),
    xaxis_title=dict(text='timestamp', font=dict(size=15)),
    yaxis_title=dict(text='numTrials', font=dict(size=15)),
    height=600,
    width=900,
    updatemenus=[dict(type='buttons',
                      buttons=[dict(label='Play',
                                    method='animate',
                                    args=[None, dict(frame=dict(duration=300, redraw=True),
                                                     fromcurrent=True)])])]
)

fig.show()

In [5]:
conn.close()